# Crime Report NLP Pipeline
**Dataset:** CrimeReport — Twitter/social-media crime reports (`CrimeReport.txt`)

**Pipeline stages:**
1. Install dependencies
2. Load & parse raw JSON-per-line data
3. Text cleaning & preprocessing (NLTK)
4. Named Entity Recognition — spaCy (`en_core_web_sm`)
5. Sentiment analysis — HuggingFace `distilbert-base-uncased-finetuned-sst-2-english`
6. Zero-shot topic classification — HuggingFace `facebook/bart-large-mnli`
7. Structured CSV export


## 1 — Install Dependencies

In [ ]:
# Run once per Colab session
!pip install -q spacy transformers torch nltk pandas
!python -m spacy download en_core_web_sm -q

## 2 — Imports & NLTK Downloads

In [ ]:
import json
import re
import html
import string
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import spacy
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from transformers import pipeline

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('All imports successful.')

## 3 — Load Raw Data
Each line of `CrimeReport.txt` is a standalone JSON object (Twitter API format).  
We extract the tweet `text`, numeric `id`, and the tweeting `user.screen_name` as the *Source*.

In [ ]:
# ── Path configuration ────────────────────────────────────────────────────────
# Running locally: adjust the path below.
# Running on Colab: upload CrimeReport.txt first, then set DATA_PATH = 'CrimeReport.txt'
DATA_PATH = 'CrimeReport.txt'   # <-- change if needed

records = []
with open(DATA_PATH, 'r', encoding='utf-8') as fh:
    for line_no, line in enumerate(fh, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            print(f'  [WARN] Line {line_no} skipped — JSON error: {e}')
            continue

        tweet_id  = str(obj.get('id', f'line_{line_no}'))
        text      = obj.get('text', '')
        user      = obj.get('user', {})
        screen    = user.get('screen_name', 'Unknown')
        name_disp = user.get('name', screen)

        records.append({
            'tweet_id'  : tweet_id,
            'screen_name': screen,
            'display_name': name_disp,
            'raw_text'  : text,
            'created_at': obj.get('created_at', ''),
        })

df_raw = pd.DataFrame(records)
print(f'Loaded {len(df_raw)} tweets.')
df_raw.head(3)

## 4 — Text Cleaning & Preprocessing
Steps applied:
- HTML entity decoding
- Remove URLs, @mentions, #hashtag symbols, RT prefix
- Lowercase + strip punctuation & extra whitespace
- NLTK tokenization → stopword removal → Porter stemming  
  *(tokens kept separately; NER & sentiment run on the cleaned but un-stemmed text)*

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
stemmer    = PorterStemmer()

# ── regex patterns ────────────────────────────────────────────────────────────
RE_URL      = re.compile(r'https?://\S+|www\.\S+')
RE_MENTION  = re.compile(r'@\w+')
RE_HASHTAG  = re.compile(r'#(\w+)')          # keep the word, drop the #
RE_RT       = re.compile(r'^RT\s+', re.IGNORECASE)
RE_SPACES   = re.compile(r'\s+')


def clean_text(raw: str) -> str:
    """Return lightly-cleaned text suitable for NER & sentiment."""
    text = html.unescape(raw)          # &amp; → &, \u2019 etc.
    text = RE_RT.sub('', text)         # drop leading RT
    text = RE_URL.sub('', text)        # remove URLs
    text = RE_MENTION.sub('', text)    # remove @mentions
    text = RE_HASHTAG.sub(r'\1', text) # #Police → Police
    text = RE_SPACES.sub(' ', text).strip()
    return text


def tokenize_stem(cleaned: str) -> list:
    """Lowercase tokenize → remove stopwords & punctuation → stem."""
    tokens = word_tokenize(cleaned.lower())
    tokens = [
        stemmer.stem(t)
        for t in tokens
        if t not in STOP_WORDS and t not in string.punctuation and len(t) > 1
    ]
    return tokens


df_raw['cleaned_text'] = df_raw['raw_text'].apply(clean_text)
df_raw['tokens']       = df_raw['cleaned_text'].apply(tokenize_stem)

print('Sample cleaned texts:')
df_raw[['raw_text', 'cleaned_text', 'tokens']].head(3)

## 5 — Named Entity Recognition (spaCy)
Label mapping used:
| spaCy label | Our category |
|---|---|
| PERSON | Person |
| GPE, LOC, FAC | Location |
| ORG | Organization |
| DATE, TIME | Date/Time |

The `Entities` column stores a semicolon-separated list of `text (TYPE)` entries.

In [ ]:
nlp = spacy.load('en_core_web_sm')

LABEL_MAP = {
    'PERSON' : 'Person',
    'GPE'    : 'Location',
    'LOC'    : 'Location',
    'FAC'    : 'Location',
    'ORG'    : 'Organization',
    'DATE'   : 'Date',
    'TIME'   : 'Date',
}


def extract_entities(text: str) -> str:
    """Run spaCy NER and return a semicolon-separated string of 'entity (TYPE)'."""
    if not text.strip():
        return ''
    doc  = nlp(text)
    seen = set()
    ents = []
    for ent in doc.ents:
        category = LABEL_MAP.get(ent.label_)
        if category is None:
            continue
        key = (ent.text.strip(), category)
        if key not in seen:
            seen.add(key)
            ents.append(f'{ent.text.strip()} ({category})')
    return '; '.join(ents) if ents else 'None'


df_raw['Entities'] = df_raw['cleaned_text'].apply(extract_entities)
print('NER complete. Sample entities:')
df_raw[['cleaned_text', 'Entities']].head(8)

## 6 — Sentiment Analysis
Model: `cardiffnlp/twitter-roberta-base-sentiment-latest`  
Trained on 124M tweets — same domain as this dataset. Natively returns `Positive`, `Neutral`, or `Negative` (no threshold workaround needed).

In [ ]:
# Truncation prevents errors on long tweets
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    truncation=True,
    max_length=512,
)


def get_sentiment(text: str) -> str:
    if not text.strip():
        return 'Neutral'
    result = sentiment_pipe(text)[0]
    label  = result['label']   # 'positive' / 'neutral' / 'negative'
    return label.capitalize()  # → 'Positive' / 'Neutral' / 'Negative'


print('Running sentiment analysis (may take ~1 min on CPU)...')
df_raw['Sentiment'] = df_raw['cleaned_text'].apply(get_sentiment)

print('\nSentiment distribution:')
print(df_raw['Sentiment'].value_counts())
df_raw[['cleaned_text', 'Sentiment']].head(5)

## 7 — Topic Classification (Zero-Shot)
Model: `facebook/bart-large-mnli`

Candidate labels:
- `Theft / Robbery`
- `Assault / Violence`
- `Fire / Arson`
- `Traffic Accident`
- `Public Disturbance`
- `Other`

The label with the highest probability is selected; if the top score is below 0.25, we fall back to `Other`.

In [ ]:
classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
)

TOPIC_LABELS = [
    'Theft / Robbery',
    'Assault / Violence',
    'Fire / Arson',
    'Traffic Accident',
    'Public Disturbance',
    'Other',
]

TOPIC_THRESHOLD = 0.25   # confidence below this → 'Other'


def classify_topic(text: str) -> str:
    if not text.strip():
        return 'Other'
    result     = classifier(text, TOPIC_LABELS, multi_label=False)
    top_label  = result['labels'][0]
    top_score  = result['scores'][0]
    return top_label if top_score >= TOPIC_THRESHOLD else 'Other'


print('Running zero-shot topic classification (may take ~3-5 min on CPU)...')
df_raw['Topic'] = df_raw['cleaned_text'].apply(classify_topic)

print('\nTopic distribution:')
print(df_raw['Topic'].value_counts())
df_raw[['cleaned_text', 'Sentiment', 'Topic']].head(5)

## 8 — Assemble Final Structured Output
Schema: `Text_ID | Source | Raw_Text | Sentiment | Entities | Topic`

- **Text_ID** — `TXT_` + zero-padded row index (e.g. `TXT_001`)
- **Source** — Twitter screen name (`@handle`), or `CrimeReport` if missing
- **Raw_Text** — original tweet text (un-cleaned)
- **Sentiment** — `Positive` / `Negative` / `Neutral`
- **Entities** — semicolon-separated NER results
- **Topic** — classified crime category

In [ ]:
df_output = pd.DataFrame({
    'Text_ID'   : ['TXT_' + str(i).zfill(3) for i in range(1, len(df_raw) + 1)],
    'Source'    : df_raw['screen_name'].apply(lambda s: f'@{s}' if s != 'Unknown' else 'CrimeReport').values,
    'Raw_Text'  : df_raw['raw_text'].values,
    'Sentiment' : df_raw['Sentiment'].values,
    'Entities'  : df_raw['Entities'].values,
    'Topic'     : df_raw['Topic'].values,
})

print(f'Output shape: {df_output.shape}')
df_output.head(10)

## 9 — Export to CSV

In [ ]:
OUTPUT_CSV = 'crime_nlp_results.csv'
df_output.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Saved → {OUTPUT_CSV}  ({len(df_output)} rows)')

# ── Auto-download if running on Google Colab ──────────────────────────────────
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print('Download triggered.')
except ImportError:
    print('Not running on Colab — file saved locally.')

## 10 — Quick Validation Preview
Peek at the first 15 rows in the final output table.

In [ ]:
pd.set_option('display.max_colwidth', 80)
df_output.head(15)

## 11 — Summary Statistics

In [ ]:
print('=== Sentiment Breakdown ===')
print(df_output['Sentiment'].value_counts().to_string())

print('\n=== Topic Breakdown ===')
print(df_output['Topic'].value_counts().to_string())

print('\n=== Records with at least one Entity ===')
has_entity = df_output['Entities'].ne('None').sum()
print(f'{has_entity} / {len(df_output)} records ({has_entity/len(df_output)*100:.1f}%)')